In [ ]:
import pandas as pd

# -----------------------------
# 1. 데이터 로드
# -----------------------------
orders = pd.read_csv('data/raw/orders.csv')
prior = pd.read_csv('data/raw/order_products__prior.csv')
products = pd.read_csv('data/raw/products.csv')
aisles = pd.read_csv('data/raw/aisles.csv')
departments = pd.read_csv('data/raw/departments.csv')

print("데이터 로드 완료")

# -----------------------------
# 2. 원본 데이터 메모리 최적화
# -----------------------------
# orders
orders['order_id'] = pd.to_numeric(orders['order_id'], downcast='integer')
orders['user_id'] = pd.to_numeric(orders['user_id'], downcast='integer')
orders['order_number'] = pd.to_numeric(orders['order_number'], downcast='integer')
orders['order_dow'] = pd.to_numeric(orders['order_dow'], downcast='integer')
orders['order_hour_of_day'] = pd.to_numeric(orders['order_hour_of_day'], downcast='integer')
orders['days_since_prior_order'] = pd.to_numeric(
    orders['days_since_prior_order'], downcast='float'
)

# prior
prior['order_id'] = pd.to_numeric(prior['order_id'], downcast='integer')
prior['product_id'] = pd.to_numeric(prior['product_id'], downcast='integer')
prior['add_to_cart_order'] = pd.to_numeric(prior['add_to_cart_order'], downcast='integer')
prior['reordered'] = pd.to_numeric(prior['reordered'], downcast='integer')

# products / aisles / departments
products['product_id'] = pd.to_numeric(products['product_id'], downcast='integer')
products['aisle_id'] = pd.to_numeric(products['aisle_id'], downcast='integer')
products['department_id'] = pd.to_numeric(products['department_id'], downcast='integer')

aisles['aisle_id'] = pd.to_numeric(aisles['aisle_id'], downcast='integer')
departments['department_id'] = pd.to_numeric(departments['department_id'], downcast='integer')

print("원본 데이터 메모리 최적화 완료")

# -----------------------------
# 3. 상품 메타데이터 병합
# -----------------------------
product_info = products.merge(aisles, on='aisle_id', how='left')
product_info = product_info.merge(departments, on='department_id', how='left')

# 문자열은 category로 변환
if 'product_name' in product_info.columns:
    product_info['product_name'] = product_info['product_name'].astype('category')
if 'aisle' in product_info.columns:
    product_info['aisle'] = product_info['aisle'].astype('category')
if 'department' in product_info.columns:
    product_info['department'] = product_info['department'].astype('category')

# -----------------------------
# 4. orders에서 prior 주문만 분리
# -----------------------------
prior_orders = orders[orders['eval_set'] == 'prior'][[
    'order_id',
    'user_id',
    'order_number',
    'order_dow',
    'order_hour_of_day',
    'days_since_prior_order'
]].copy()

# -----------------------------
# 5. prior에 user / order 정보 붙이기
# -----------------------------
prior = prior.merge(prior_orders, on='order_id', how='left')

# -----------------------------
# 6. 유저-상품별 과거 구매 이력 피처 생성
# -----------------------------
user_product_features = prior.groupby(['user_id', 'product_id']).agg(
    up_order_count=('order_id', 'count'),
    up_first_order_num=('order_number', 'min'),
    up_last_order_num=('order_number', 'max'),
    up_avg_add_to_cart=('add_to_cart_order', 'mean'),
    up_reordered_sum=('reordered', 'sum'),
    up_reordered_mean=('reordered', 'mean'),
    up_reordered_last=('reordered', 'last')
).reset_index()

# -----------------------------
# 7. 유저별 전체 주문 수 피처
# -----------------------------
user_features = prior.groupby('user_id').agg(
    user_total_orders=('order_number', 'max'),
    user_total_items=('product_id', 'count'),
    user_total_distinct_items=('product_id', 'nunique'),
    user_avg_days_since_prior=('days_since_prior_order', 'mean')
).reset_index()

# -----------------------------
# 8. 상품별 피처
# -----------------------------
product_features = prior.groupby('product_id').agg(
    product_total_purchases=('order_id', 'count'),
    product_total_reorders=('reordered', 'sum'),
    product_unique_users=('user_id', 'nunique')
).reset_index()

product_features['product_reorder_rate'] = (
    product_features['product_total_reorders'] / product_features['product_total_purchases']
)

# -----------------------------
# 9. 학습용 base 생성 대신
#    prior 기준 유저-상품 조합 생성
# -----------------------------
df = prior[['user_id', 'product_id']].drop_duplicates().copy()

# -----------------------------
# 10. 피처들 병합
# -----------------------------
df = df.merge(user_product_features, on=['user_id', 'product_id'], how='left')
df = df.merge(user_features, on='user_id', how='left')
df = df.merge(product_features, on='product_id', how='left')

# product_info가 꼭 필요할 때만 사용
# df = df.merge(product_info, on='product_id', how='left')

# -----------------------------
# 11. 추가 파생변수
# -----------------------------
df['up_order_rate'] = df['up_order_count'] / df['user_total_orders']
df['up_recency'] = df['user_total_orders'] - df['up_last_order_num']
df['recency_ratio'] = df['up_recency'] / (df['user_total_orders'] + 1)
df['reorder_prob'] = df['up_reordered_sum'] / (df['up_order_count'] + 1)
df['user_product_preference'] = df['up_order_rate'] * df['reorder_prob']

# -----------------------------
# 12. 결측치 처리
# -----------------------------
df['up_order_count'] = df['up_order_count'].fillna(-1)
df['up_recency'] = df['up_recency'].fillna(-1)
df['recency_ratio'] = df['recency_ratio'].fillna(-1)
df['user_product_preference'] = df['user_product_preference'].fillna(-1)

# -----------------------------
# 13. 필요한 컬럼만 선택
# -----------------------------
final_df = df[
    [
        'user_id',
        'product_id',
        'up_order_count',
        'up_recency',
        'recency_ratio',
        'user_product_preference',
        'up_reordered_sum',
        'up_reordered_mean',
        'up_reordered_last'
    ]
].copy()

# -----------------------------
# 14. 최종 메모리 최적화
# -----------------------------
print(
    "최적화 전 메모리 사용량:",
    round(final_df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

int_cols = ['user_id', 'product_id', 'up_order_count', 'up_recency']
for col in int_cols:
    final_df[col] = pd.to_numeric(final_df[col], downcast='integer')

float_cols = ['recency_ratio', 'user_product_preference']
for col in float_cols:
    final_df[col] = pd.to_numeric(final_df[col], downcast='float')
    final_df[col] = final_df[col].round(4)

print(
    "최적화 후 메모리 사용량:",
    round(final_df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

print(final_df.dtypes)

# -----------------------------
# 15. CSV 저장
# -----------------------------
final_df.to_csv(
    'data/user_product_core_features.csv',
    index=False,
    encoding='utf-8-sig',
    float_format='%.4f'
)

print("전처리 완료")
print("최종 데이터 크기:", final_df.shape)
print(final_df.head())